# SQL Agent

This notebook builds a LangChain SQL Agent that can answer questions using data stored in Snowflake.

### Imports

In [1]:
from dotenv import load_dotenv
import os

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

C:\Users\PMLS\AppData\Local\Temp\ipykernel_24860\178467958.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


### Snowflake Connection

Load Snowflake credentials from the `.env` file and establish a connection to the OLIST database.

In [2]:
load_dotenv()

account = os.getenv("SNOWFLAKE_ACCOUNT")
user = os.getenv("SNOWFLAKE_USER")
password = os.getenv("SNOWFLAKE_PASSWORD")
warehouse = os.getenv("SNOWFLAKE_WAREHOUSE")
database = os.getenv("SNOWFLAKE_DATABASE")
schema = os.getenv("SNOWFLAKE_SCHEMA")

connection_url = (
    f"snowflake://{user}:{password}@{account}/"
    f"{database}/{schema}?warehouse={warehouse}")

db = SQLDatabase.from_uri(connection_url)

In [3]:
db = SQLDatabase.from_uri(connection_url)


In [4]:
print(db.dialect)
print(db.get_usable_table_names())

snowflake
['customers', 'order_items', 'order_payments', 'orders', 'product_category_translation', 'products']


### LLM

In [5]:
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0
)

### ToolKit

In [6]:
toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

tools = toolkit.get_tools()

for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


### SQL Agent

In [7]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a SQL database agent.

Follow this workflow strictly:

1. First, use sql_db_list_tables to see the available tables.
2. After listing the tables, ALWAYS use sql_db_schema to inspect
   the schema of the relevant table.
3. NEVER write SQL before inspecting the relevant table schema.
4. NEVER guess table names or column names.
5. When using sql_db_schema, ALWAYS use the argument `table_names`,
   not `table`.
6. Generate SQL only using table and column names found in the schema.
7. Use sql_db_query_checker to check the SQL query.
8. After checking the SQL, ALWAYS use sql_db_query to execute it.
9. NEVER invent, estimate, or assume a query result.
10. Give the final answer only from the result returned by
    sql_db_query.
11. Follow the exact input schema of every tool.
"""
)

In [8]:
example_query = "What is the average payment value across all orders?"

events = agent.stream(
    {
        "messages": [
            ("user", example_query)
        ]
    },
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the average payment value across all orders?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (eab91a3a-b2f6-4c07-9bc4-9fe63fd83f38)
 Call ID: eab91a3a-b2f6-4c07-9bc4-9fe63fd83f38
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

customers, order_items, order_payments, orders, product_category_translation, products
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (cbc1fb44-1a57-4d44-94eb-c0f9d9e8f286)
 Call ID: cbc1fb44-1a57-4d44-94eb-c0f9d9e8f286
  Args:
    table_names: order_payments
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE order_payments (
	order_id VARCHAR(16777216), 
	payment_sequential DECIMAL(38, 0), 
	payment_type VARCHAR(167772